# Fetal Planes – Initial Data Exploration

In [ ]:
import os
from collections.abc import Callable, Iterable, Iterator, Sequence
from math import ceil, sqrt
from typing import List, Literal, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import torch
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import ConcatDataset, Dataset
from torchvision.io import read_image
from tqdm import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.dataset import FetalBrainPlanesDataset, FetalPlanesDataset
from src.data.utils import group_split
from src.data.utils.utils import show_pytorch_images

data_dir = root / "data"
dataset_name = "FETAL_PLANES"
dataset_dir = data_dir / dataset_name

## Class Balance and Split Exploration

Load the dataset CSV and examine class distributions, then explore patient-aware splitting strategies to achieve balanced train / val / test sets.

In [ ]:
img_labels = pd.read_csv(f"{dataset_dir}/data.csv")
img_labels
# 12400 rows × 7 columns

### Class Weights

Compute inverse-frequency class weights for each `Brain_plane` label.
These weights are passed to the loss function to compensate for class imbalance during training.

In [ ]:
brain_plane = img_labels["Brain_plane"]
counts = brain_plane.value_counts(sort=False).sort_index()

classes = np.unique(brain_plane)
class_weight = compute_class_weight(class_weight="balanced", classes=classes, y=brain_plane)

pd.DataFrame({"Count": counts[classes], "Weight": class_weight})

### Dataset Split – First Pass

Instantiate `FetalBrainPlanesDataset` over the full dataset and apply a 90/10 patient-group split (seed 6106).
Print per-class sample counts for each split to check balance.

In [ ]:
def get_class_num(dataset: Dataset) -> Sequence[int]:
    classes = torch.tensor([FetalBrainPlanesDataset.labels.index(dataset[i, 1]) for i in range(len(dataset))])
    classes_indices = [torch.nonzero(classes == class_id).flatten() for class_id in torch.arange(5)]
    classes_num_samples = [len(indices) for indices in classes_indices]
    return classes_num_samples


data_all = FetalBrainPlanesDataset(
    data_dir=data_dir,
    subset="all",
)
data_train, data_test = group_split(
    dataset=data_all,
    test_size=0.1,
    groups=data_all.img_labels["Patient_num"],
    random_state=6106,  # 0.1: [8190, 2749, 6106, 8394, 9592, 1585, 990, 2520, 8838, 6802]
)

print(len(data_all))
print(get_class_num(data_all))

print(len(data_test), len(data_test) / len(data_all))
print(get_class_num(data_test))

print(len(data_train), len(data_train) / len(data_all))
print(get_class_num(data_train))

### Dataset Split – Nested Train / Val

Refine the split: take the 90 % training portion and further divide it into ~80 % train and ~10 % val using a second `GroupShuffleSplit` (seed 6277).
Print class counts per subset.

In [ ]:
data_all = FetalBrainPlanesDataset(
    data_dir=data_dir,
    subset="all",
)
data_train, data_test = group_split(
    dataset=data_all,
    test_size=0.1,
    groups=data_all.img_labels["Patient_num"],
    random_state=6106,  # 0.1: [8190, 2749, 6106, 8394, 9592, 1585, 990, 2520, 8838, 6802]
)

data_train, data_val = group_split(
    dataset=data_train,
    test_size=0.114,
    groups=data_all.img_labels.iloc[data_train.indices].reset_index(drop=True)["Patient_num"],
    random_state=6277,  # [7539, 6277, 2613, 6652, 2769, 653, 1084, 3368, 9111, 9101]
)

print(len(data_test), len(data_test) / len(data_all))
print(get_class_num(data_test))

print(len(data_val), len(data_val) / len(data_all))
print(get_class_num(data_val))

print(len(data_train), len(data_train) / len(data_all))
print(get_class_num(data_train))

## Image Resolution Analysis

Read the spatial dimensions (height × width) of every image in the dataset and plot histograms with mean and median markers.
The median aspect ratio is used to derive target crop resolutions — e.g. 192 × 256 — for downstream model training.

In [ ]:
def extend_mean_median_plot(ax, mean, median, title):
    ax.axvline(mean, color="r", linestyle="dashed", linewidth=2)
    ax.axvline(median, color="b", linestyle="dashed", linewidth=2)
    ax.legend([f"mean {mean}", f"median {median}"], loc="upper left")
    ax.set_title(title, fontsize=14)


def print_resolution(scale, width):
    height = width / scale
    print(f"{height:.2f} / {width}")


data = []
for index, row in tqdm(img_labels.iterrows(), total=len(img_labels), desc="Read images shape"):
    image_name = row["Image_name"]
    image_path = dataset_dir / "Images" / f"{image_name}.png"
    image = read_image(image_path)
    data.append((image.shape[0], image.shape[1], image.shape[2]))

df_shape = pd.DataFrame(data=data)
mean = df_shape.mean()
median = df_shape.median()

axs = df_shape.hist(column=[1, 2], bins=100, figsize=(20, 5))
extend_mean_median_plot(axs[0][0], mean[1], median[1], "Height")
extend_mean_median_plot(axs[0][1], mean[2], median[2], "Width")

scale = median[2] / median[1]
print_resolution(scale, 64)  # 64 / 64
print_resolution(scale, 128)  # 96 / 128
print_resolution(scale, 256)  # 192 / 256

## Image Mean and Std Analysis

Compute grayscale channel mean and standard deviation across the full dataset at several resize scales.
The resulting values (~0.17 mean, ~0.19 std) are used as normalisation constants for the data pipeline.

In [ ]:
def get_mean_std(dataset):
    # var[X] = E[X**2] - E[X]**2
    channels_sum = 0
    channels_sqrd_sum = 0

    for data, _ in tqdm(dataset):
        channels_sum += torch.mean(data, dim=[1, 2])
        channels_sqrd_sum += torch.mean(data**2, dim=[1, 2])

    mean = channels_sum / len(dataset)
    std = (channels_sqrd_sum / len(dataset) - mean**2) ** 0.5

    return mean, std


def find_mean_std(height, width):
    train = FetalPlanesDataset(
        data_dir=data_dir,
        train=True,
        transform=torch.nn.Sequential(
            T.Grayscale(),
            T.Resize((height, width)),
            T.ToDtype(torch.float32, scale=True),
        ),
    )

    mean, std = get_mean_std(train)
    print(f"For {height} / {width} mean is {mean.item():.2f} std is {std.item():.2f}")


# print(mean.item())  # 0.16958117485046387, 0.16958120197261892
# print(std.item())  # 0.1906554251909256,  0.19065533816416103
find_mean_std(55, 80)  # 0.17 / 0.19
find_mean_std(165, 240)  # 0.17 / 0.19
find_mean_std(415, 600)  # 0.17 / 0.19

## Split Seed Analysis

Scan 10 000 candidate `GroupShuffleSplit` seeds and score each by how evenly it distributes brain-plane classes across train and val.
The top 10 seeds are plotted; seed 5724 is selected for a 20 % val split used in all experiments.

In [ ]:
def group_split_label(
    dataset: pd.DataFrame, test_size: float, groups, random_state: int = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    splitter = GroupShuffleSplit(test_size=test_size, n_splits=1, random_state=random_state)
    split = splitter.split(dataset, groups=groups)
    train_idx, test_idx = next(split)
    return dataset.iloc[train_idx], dataset.iloc[test_idx]


def get_similarity(train, test, test_size):
    similarity = 0
    train_count = train.value_counts(sort=False).sort_index()
    test_count = test.value_counts(sort=False).sort_index()

    if train_count.index.tolist() != test_count.index.tolist():
        return -1

    for a, b in zip(train_count, test_count):
        similarity += (a * test_size - b * (1 - test_size)) ** 2
    return similarity**0.5


def plt_value_counts(ax, dataset, tile=None):
    counts = dataset.value_counts(sort=False).sort_index()
    counts.plot(kind="bar", ax=ax)
    if tile:
        ax.set_title(tile)


def plt_group_split(dataset: pd.DataFrame, test_size: float, random_states: List[int], top_states: int = None):
    splits = []
    for random_state in tqdm(random_states):
        tr, val = group_split_label(
            dataset,
            test_size=test_size,
            groups=dataset["Patient_num"],
            random_state=random_state,
        )

        similarity = get_similarity(tr.Brain_plane, val.Brain_plane, test_size)
        if similarity >= 0:
            splits.append((similarity, tr, val, random_state))

    splits.sort(key=lambda e: (e[0], e[3]))
    nrows = top_states if top_states else len(splits)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=2,
        sharex="all",
        squeeze=False,
        figsize=(20, 3 * nrows),
    )
    fig.suptitle(f"Test size {test_size}")
    for i, (similarity, tr, val, random_state) in enumerate(splits[:nrows]):
        plt_value_counts(axes[i, 0], tr.Brain_plane, tile=f"Seed {random_state}")
        plt_value_counts(axes[i, 1], val.Brain_plane, tile=f"Similarity {similarity}")

    plt.show()
    print([random_state for (similarity, tr, val, random_state) in splits[:nrows]])


train = FetalPlanesDataset(
    data_dir=data_dir,
    train=True,
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.Resize((165, 240)),
        T.ToDtype(torch.float32, scale=True),
    ),
)


plt_group_split(
    train.img_labels,
    test_size=0.2,
    random_states=list(range(10000)),
    top_states=10,
)  # 564, 3097, 1683, 4951, 5724, 8910, 9486, 7023, 5907, 9759
# plt_group_split(
#     train.img_labels,
#     test_size=0.15,
#     random_states=list(range(10000)),
#     top_states=10,
# )  # 943, 9787, 4935, 6588, 6893, 697, 6347, 5785, 4, 7765
# plt_group_split(
#     train.img_labels,
#     test_size=0.1,
#     random_states=list(range(10000)),
#     top_states=10,
# )  # 2251, 3084, 9456, 8902, 1208, 9959, 2696, 2086, 4063, 9126